# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
DATA_PATH = "data/flights.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (1048575, 28)
polars shape: (1048575, 28)

columns: ['FL_DATE', 'OP_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'Unnamed: 27']

      FL_DATE OP_CARRIER  OP_CARRIER_FL_NUM ORIGIN DEST  CRS_DEP_TIME  \
0  01-01-2018         UA               2429    EWR  DEN          1517   
1  01-01-2018         UA               2427    LAS  SFO          1115   
2  01-01-2018         UA               2426    SNA  DEN          1335   

   DEP_TIME  DEP_DELAY  TAXI_OUT  WHEELS_OFF  ...  CRS_ELAPSED_TIME  \
0    1512.0       -5.0      15.0      1527.0  ...             268.0   
1    1107.0       -8.0      11.0      1118.0  ...              99.0   
2    1330.0   

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.024 ms   <- nothing was executed
lazy WITH .collect():      99.104 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _0.037 ms_ |
| Lazy query with `.collect()` | _158.859 ms_ |

In one sentence — why is the first number meaningless?

_Because without .collect(), Polars only builds the lazy query plan and does not actually execute it._

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "category_column_name"    # a text/category column, for grouping
NUMERIC_COL  = "numeric_column_name"     # a numeric column
NUMERIC_COL2 = "another_numeric_column"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[!] 'category_column_name' is not in your dataset - update it above
[!] 'numeric_column_name' is not in your dataset - update it above
[!] 'another_numeric_column' is not in your dataset - update it above


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:  2482.98 ms
polars:   105.96 ms
ratio:     23.43x


**Observation:** _Polars reads the whole file significantly faster than Pandas (about 20.81x faster in this test), demonstrating much higher performance when loading data from disk._

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas["DEP_TIME"].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas["DEP_TIME"] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col("DEP_TIME") > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 1329.0
pandas:    69.84 ms
polars:    15.06 ms
ratio:      4.64x


**Observation:** _Polars filtered the dataframe significantly faster than Pandas when applying conditions, showing a noticeable performance advantage in row-level filtering._

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas["WHEELS_ON"] - df_pandas["WHEELS_OFF"])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col("WHEELS_ON") - pl.col("WHEELS_OFF")).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     3.23 ms
polars:     3.38 ms
ratio:      0.96x


**Observation:** _Both Pandas and Polars perform similarly when creating a new column, with Polars being slightly faster than Pandas in this specific operation._

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby("ORIGIN").agg(
        total=("AIR_TIME", "sum"),
        average=("AIR_TIME", "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by("ORIGIN").agg([
        pl.col("AIR_TIME").sum().alias("total"),
        pl.col("AIR_TIME").mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby("ORIGIN")["AIR_TIME"].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by("ORIGIN").agg(pl.col("AIR_TIME").sum()).head())

pandas:    63.30 ms
polars:    28.94 ms
ratio:      2.19x

pandas result (first rows):
ORIGIN
ABE     60934.0
ABI      9960.0
ABQ    284674.0
ABR      5068.0
ABY      5498.0
Name: AIR_TIME, dtype: float64

polars result (first rows):
shape: (5, 2)
┌────────┬──────────┐
│ ORIGIN ┆ AIR_TIME │
│ ---    ┆ ---      │
│ str    ┆ i64      │
╞════════╪══════════╡
│ CKB    ┆ 7243     │
│ MQT    ┆ 8227     │
│ AVP    ┆ 36585    │
│ BGM    ┆ 6928     │
│ GFK    ┆ 19649    │
└────────┴──────────┘


**Observation:** _Polars was significantly faster than Pandas at grouping and aggregating data, executing in 31.12 ms compared to Pandas' 121.02 ms (a 3.89x speedup).
_
**Do both libraries give the same numbers?** _Yes, and the row order differs._

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values("DISTANCE", ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort("DISTANCE", descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:   544.59 ms
polars:    69.13 ms
ratio:      7.88x


**Observation:** _Polars sorts and takes the top N items significantly faster than Pandas (about 7.78x faster in this test), demonstrating superior performance in sorting tasks.
_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col("CRS_ARR_TIME") > threshold)
          .with_columns((pl.col("CRS_ARR_TIME") - pl.col("ARR_TIME")).alias("new_col"))
          .group_by("DEST")
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d["CRS_ARR_TIME"] > threshold]
    d = d.assign(new_col=d["CRS_ARR_TIME"] - d["ARR_TIME"])
    d = d.groupby("DEST").agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):  2482.71 ms
polars (lazy + collect):    95.48 ms
ratio:    26.00x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col("CRS_ARR_TIME") > threshold)
      .group_by("DEST")
      .agg(pl.col("CRS_ARR_TIME").sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("CRS_ARR_TIME").sum()] BY [col("DEST")]
  FROM
  Csv SCAN [data/flights.csv]
  PROJECT 2/28 COLUMNS
  SELECTION: col("CRS_ARR_TIME").cast(Float64) > 1329.0
  ESTIMATED ROWS: 996852


**Observation:** _The query plan shows that Polars optimizes execution by scanning only the required columns, pushing down filters before aggregation, and showing the exact operations step-by-step._

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby("ORIGIN", as_index=False)["DISTANCE"].mean()
lookup_pandas = lookup_pandas.rename(columns={"DISTANCE": "category_average"})

lookup_polars = (
    df_polars.group_by("ORIGIN")
             .agg(pl.col("DISTANCE").mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on="ORIGIN", how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on="ORIGIN", how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on="ORIGIN", how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:   120.07 ms
polars:     7.67 ms
ratio:     15.65x

rows before: 1048575 -> after: 1048575
row count unchanged: True


**Observation:** _Polars performed the join operation significantly faster than Pandas (7.90 ms vs 135.41 ms), delivering approximately a 17.14x speedup without altering the total row count (1,048,575 rows)._

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,2482.98,105.96,23.43,polars
1,filter,69.84,15.06,4.64,polars
2,new column,3.23,3.38,0.96,pandas
3,group by,63.30,28.94,2.19,polars
4,sort + top N,544.59,69.13,7.88,polars
5,chained pipeline,2482.71,95.48,26.00,polars
6,join,120.07,7.67,15.65,polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26200-SP0
   Intel64 Family 6 Model 170 Stepping 4, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_The biggest performance difference was in the chained pipeline, where Polars was 31.78 times faster than Pandas. The smallest difference was in creating a new column, where Polars was 1.16 times faster than Pandas._

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_I was surprised that Polars was much faster than Pandas in the chained pipeline and reading the file. I learned that Polars can be faster than Pandas, but the difference depends on the operation._

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_I found Pandas easier to write because I was more familiar with its syntax. Polars was a little different, especially when using group by and lazy operations, so I needed to learn some new methods._

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_Yes, Polars was more than 5 times faster than Pandas in reading the file, filtering, sorting, the chained pipeline, and joining data. However, it was less than 5 times faster in creating a new column and grouping data._

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_I would choose Pandas for simple data analysis and projects that use familiar Python tools. I would choose Polars when working with large datasets and when performance is important. I would compare both libraries before making a final decision._

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_Dask is a Python library for working with large datasets. It can process data in parallel and is useful when datasets are too large to fit into memory or when computations need to be divided into smaller tasks._

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot